# tiny-log-parser — live demo

A Qwen3-4B fine-tune plus a deterministic epoch pre-pass, normalizing messy log
lines into a canonical 7-field JSON record. **100% vs 83.5%** exact match against
`gemini-3.1-pro-preview` on a 200-example held-out test set.

**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all.

Repo: https://github.com/arshirazi97/tiny-log-parser

## 1. Setup (~2 min)

In [ ]:
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
%cd tiny-log-parser
!pip install -q transformers peft accelerate bitsandbytes openai

## 2. Load the model
Base weig5 GB) plus the LoRA adapter (132 MB).

In [ ]:
import torch, json, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from eval import build_prompt, parse, FIELDS
from score_hybrid import epoch_override

BASE = 'unsloth/qwen3-4b-unsloth-bnb-4bit'
tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE, device_map='auto'),
    'arshirazi/tiny-log-parser').eval()
print('ready')

## 3. The pipeline
The pre-pass fires only on bare epoch integers — the one subtask the model
reliably fails. Everything else goes to the model.

In [ ]:
def normalize(lines, batch=8):
    out = []
    for i in range(0, len(lines), batch):
        chunk = lines[i:i+batch]
        enc = tok([build_prompt(l, []) for l in chunk],
                  return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=120, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for l, o in zip(chunk, gen):
            p = parse(tok.decode(o[enc.input_ids.shape[-1]:], skip_special_tokens=True))
            iso = epoch_override(l)
            if p and iso: p = {**p, 'timestamp': iso}
            out.append(p)
    return out

def show(lines):
    t0 = time.time(); preds = normalize(lines); dt = time.time()-t0
    for l, p in zip(lines, preds):
        print('\nIN  ', l[:96], '  [pre-pass]' if epoch_override(l) else '')
        print('OUT ', json.dumps(p) if p else '[unparseable]')
    print(f'\n{len(lines)} lines in {dt:.1f}s ({dt/len(lines)*1000:.0f} ms/line)')

## 4. Six formats, one schema

In [ ]:
show([
  '<131>Mar  5 02:10:12 web-07 payments[4471]: TLS handshake aborted by peer [tid=8f2c91aa04bd77e35c1d6b0392ef4a18] took=4.775s',
  '10.14.2.9 - - [22/Jul/2026:09:15:44 +0500] "GET /api/v2/orders HTTP/1.1" 503 812 "-" "curl/8.4.0" rt=7.881',
  'ts=1780543196 level=warn service=inventory msg="stock below threshold" trace=7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e latency=340ms',
  '2026-06-18 14:22:09,331 ERROR [payment-worker-3] c.a.p.RefundService - refund gateway timeout traceId=b2e4f6a8c0d2e4f6a8b0c2d4e6f8a0b2 elapsed=2140ms',
  '{"time":1783402811,"severity":"CRITICAL","container":"billing","message":"ledger write failed","duration_us":9120000}',
  '[2026-07-15T03:44:12+05:00] [FATAL] [search-indexer] shard replication halted (tid=e9d8c7b6a5f4e3d2c1b0a9f8e7d6c5b4) 15.2s',
])

## 5. Why the pre-pass exists
The model gets minutes and seconds right and the date wrong — epoch → calendar
arithmetic is integer division it can't do. Scaling training data 5k → 20k moved
this 0.5 points, so it was routed to `datetime.fromtimestamp()` instead.

In [ ]:
line = 'ts=1780543196 level=warn service=inventory msg="stock below threshold" latency=340ms'
enc = tok(build_prompt(line, []), return_tensors='pt').to(model.device)
g = model.generate(**enc, max_new_tokens=120, do_sample=False, pad_token_id=tok.pad_token_id)
raw = parse(toke(g[0][enc.input_ids.shape[-1]:], skip_special_tokens=True))
print('model alone :', raw.get('timestamp'))
print('pre-pass    :', epoch_override(line))

## 6. Your own logs
Paste any lines below. Formats outside the six the model was trained on will
degrade — see Limitations in the README.

In [ ]:
show([
  '<134>Jul 12 18:44:03 api-02 checkout[992]: order placed successfully [tid=c41d9a7e6b28f05134ae8d90bb17e2c6] took=0.234s',
])